In [ ]:
!pip install -q segmentation_models_pytorch rasterio albumentations


In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_IMG = "/content/drive/MyDrive/satellite data/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/images"
DRIVE_LBL = "/content/drive/MyDrive/satellite data/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/labels"

# Copy to local SSD for faster I/O during training
!mkdir -p /content/local_data/images /content/local_data/labels
print("Syncing data to local SSD...")
!cp -rn "$DRIVE_IMG/." /content/local_data/images/
!cp -rn "$DRIVE_LBL/." /content/local_data/labels/
print("Sync Complete.")


In [ ]:
import glob
import os
from sklearn.model_selection import train_test_split

# Get all image and label paths
all_img_paths = sorted(glob.glob("/content/local_data/images/*.tif"))
all_lbl_paths = sorted(glob.glob("/content/local_data/labels/*.tif"))

# Extract base filenames (without path and extension)
img_basenames = {os.path.basename(p).split('.')[0] for p in all_img_paths}
lbl_basenames = {os.path.basename(p).split('.')[0] for p in all_lbl_paths}

# Find common basenames (images that have matching labels)
common_basenames = sorted(list(img_basenames.intersection(lbl_basenames)))

# Filter paths to include only those with common basenames
all_img_files = []
all_lbl_files = []

for basename in common_basenames:
    img_path = f"/content/local_data/images/{basename}.tif"
    lbl_path = f"/content/local_data/labels/{basename}.tif"
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        all_img_files.append(img_path)
        all_lbl_files.append(lbl_path)

# Split: 80% Train, 10% Val, 10% Test
train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    all_img_files, all_lbl_files, test_size=0.2, random_state=42
)
val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42
)

print(f"Total images: {len(all_img_files)}")
print(f"Training images: {len(train_imgs)}")
print(f"Validation images: {len(val_imgs)}")
print(f"Test images: {len(test_imgs)}")


In [ ]:
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset
import albumentations as A

class SlumDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        # Read Image (3 bands)
        with rasterio.open(self.images[i]) as src:
            image = src.read([1, 2, 3]).transpose(1, 2, 0)  # H, W, C
            image = (image / (image.max() + 1e-8) * 255).astype('uint8')  # Normalize to 8-bit

        # Read Mask
        with rasterio.open(self.labels[i]) as src:
            mask = src.read(1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        # To PyTorch Tensor (C x H x W), scale to [0, 1]
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()
        return image, mask


In [ ]:
import albumentations as A
import torch
import segmentation_models_pytorch as smp

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), rotate=(-45, 45), translate_percent=(-0.0625, 0.0625), p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

val_test_transform = A.Compose([
    A.Resize(256, 256),
])

BATCH_SIZE = 8

# Create Datasets
train_dataset = SlumDataset(train_imgs, train_lbls, transform=train_transform)
val_dataset   = SlumDataset(val_imgs,   val_lbls,   transform=val_test_transform)
test_dataset  = SlumDataset(test_imgs,  test_lbls,  transform=val_test_transform)

# Create DataLoaders
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


In [ ]:
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# VGG16-FPN model for binary segmentation
model = smp.FPN(
    encoder_name="vgg16",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation='sigmoid'
).to(device)

# Hybrid Loss: BCE + Dice
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()
        self.dice = smp.losses.DiceLoss(mode='binary')

    def forward(self, pred, target):
        target_float = target.float().unsqueeze(1)  # (B, 1, H, W)
        return self.bce(pred, target_float) + self.dice(pred, target_float)

criterion = HybridLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
from tqdm import tqdm

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

EPOCHS = 30
ACCUMULATION_STEPS = 4
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for i, (imgs, msks) in enumerate(pbar):
        imgs, msks = imgs.to(device), msks.to(device)
        preds = model(imgs)
        loss = criterion(preds, msks) / ACCUMULATION_STEPS
        loss.backward()

        if (i + 1) % ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS
        pbar.set_postfix(
            loss=f"{loss.item() * ACCUMULATION_STEPS:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.6f}"
        )

    # Flush any remaining gradients
    if (i + 1) % ACCUMULATION_STEPS != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_loader)


    model.eval()
    val_stats = []

    with torch.no_grad():
        for v_imgs, v_msks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            v_imgs, v_msks = v_imgs.to(device), v_msks.to(device)
            v_preds = model(v_imgs)
            stats = smp.metrics.get_stats(
                v_preds, v_msks.unsqueeze(1), mode='binary', threshold=0.5
            )
            val_stats.append(stats)

    tp = torch.cat([s[0] for s in val_stats]).sum()
    fp = torch.cat([s[1] for s in val_stats]).sum()
    fn = torch.cat([s[2] for s in val_stats]).sum()
    tn = torch.cat([s[3] for s in val_stats]).sum()
    val_f1 = smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro')

    scheduler.step(val_f1)

    print(f">> Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val F1: {val_f1.item():.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'vggnew_slum_model.pth')
        print(f" vgg new Model Saved (F1: {best_f1:.4f}) ***")

print(f"\nTraining complete. Best Val F1: {best_f1:.4f}")


In [ ]:
# Load best weights
model.load_state_dict(torch.load('vggnew_slum_model.pth', map_location=device))
model.eval()

test_stats = []
with torch.no_grad():
    for t_imgs, t_msks in tqdm(test_loader, desc="Test Evaluation"):
        t_imgs, t_msks = t_imgs.to(device), t_msks.to(device)
        t_preds = model(t_imgs)
        stats = smp.metrics.get_stats(
            t_preds, t_msks.unsqueeze(1), mode='binary', threshold=0.5
        )
        test_stats.append(stats)

tp = torch.cat([s[0] for s in test_stats]).sum()
fp = torch.cat([s[1] for s in test_stats]).sum()
fn = torch.cat([s[2] for s in test_stats]).sum()
tn = torch.cat([s[3] for s in test_stats]).sum()

test_f1        = smp.metrics.f1_score(tp, fp, fn, tn, reduction='micro')
test_precision = (tp / (tp + fp + 1e-8)).item()
test_recall    = (tp / (tp + fn + 1e-8)).item()
test_accuracy  = ((tp + tn) / (tp + fp + fn + tn + 1e-8)).item()
test_iou       = smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro')

print(f"  TEST SET RESULTS")
print(f"  F1 Score  : {test_f1.item():.4f}")
print(f"  IoU       : {test_iou.item():.4f}")
print(f"  Precision : {test_precision:.4f}")
print(f"  Recall    : {test_recall:.4f}")
print(f"  Accuracy  : {test_accuracy:.4f}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_results(model, test_loader, device, num_samples=50):
    """Display satellite image, ground truth mask, and model prediction side by side."""
    model.eval()
    samples_shown = 0

    with torch.no_grad():
        for imgs, msks in test_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            preds = model(imgs)

            imgs_cpu  = imgs.cpu()
            msks_cpu  = msks.cpu()
            preds_cpu = preds.cpu()

            for i in range(imgs_cpu.shape[0]):
                if samples_shown >= num_samples:
                    return

                fig, axes = plt.subplots(1, 3, figsize=(15, 5))

                # Original image — already in [0, 1] range, just permute to H,W,C
                img_display = imgs_cpu[i].permute(1, 2, 0).numpy()
                img_display = np.clip(img_display, 0, 1)

                axes[0].imshow(img_display)
                axes[0].set_title("Satellite Image")
                axes[0].axis('off')

                # Ground truth mask
                axes[1].imshow(msks_cpu[i], cmap='Reds')
                axes[1].set_title("Ground Truth")
                axes[1].axis('off')

                # Model prediction (thresholded)
                pred_mask = (preds_cpu[i][0] > 0.5).float()
                axes[2].imshow(pred_mask, cmap='Blues')
                axes[2].set_title(f"Prediction (Best F1: {best_f1:.4f})")
                axes[2].axis('off')

                plt.tight_layout()
                plt.show()
                samples_shown += 1

visualize_results(model, test_loader, device, num_samples=50)


In [ ]:
import torch
import rasterio
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A


WEIGHTS_PATH = "/content/vggnew_slum_model.pth"
IMAGE_DIR    = "/content/images"
THRESHOLD    = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


inf_model = smp.FPN(
    encoder_name="vgg16",
    encoder_weights=None,  # no need to download ImageNet weights again
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(device)

inf_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
inf_model.eval()
print("Inference model loaded.")


def load_image(tif_path):
    """Load a GeoTIFF, take the first 3 bands, normalize, resize to 256x256."""
    with rasterio.open(tif_path) as src:
        img = src.read()[:3]                # first 3 channels, shape (C, H, W)
        img = img.transpose(1, 2, 0)        # H, W, C
        img = (img / (img.max() + 1e-8)).astype(np.float32)  # normalize to [0, 1]

    img = A.Resize(256, 256)(image=img)["image"]
    tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)
    vis    = img  # already H,W,C float32 in [0,1]
    return tensor, vis


tif_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.tif")))
print(f"Found {len(tif_files)} .tif files in '{IMAGE_DIR}'")

for tif in tif_files:
    img_tensor, img_vis = load_image(tif)

    with torch.no_grad():
        pred = inf_model(img_tensor)

    mask = (pred.squeeze().cpu().numpy() > THRESHOLD).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(np.clip(img_vis, 0, 1))
    axes[0].set_title(f"Input: {os.path.basename(tif)}")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Predicted Slum Mask")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
